# Ray Serve model multiplexing: one pool, many checkpoints

A small shop serves one **base model** to every chemist, plus one **specialist per program**, all from a
single GPU. Ray Serve **model multiplexing** is the API for that: one deployment holds several models by
id, loads each on demand, and evicts the least-used when it is full.

This notebook runs the **actual Ray Serve API** in the Workspace, with no service deploy and no HTTP
wrapper, so every call is visible:

- `@serve.multiplexed(max_num_models_per_replica=...)` on the loader,
- `serve.get_multiplexed_model_id()` inside the deployment,
- `serve.run(...)` to start it and get a `DeploymentHandle`,
- `handle.options(multiplexed_model_id="...").remote(...)` to pick a model per request.

The `ModelPool` below is deliberately tiny (it loads a checkpoint path, not a GPU model) so the whole
notebook runs anywhere. `serve_viewer_multi.py` in this repo is the same pattern loading real OpenFold3
checkpoints.

## 1. The multiplexed deployment

`get_model` is the loader: Ray calls it **only on a cache miss** and keeps the result per id, so each
line it prints is a real load. `__call__` reads the requested id with `serve.get_multiplexed_model_id()`
and awaits the cached model. `serve.get_replica_context().replica_tag` shows which replica served the
request.

In [ ]:
import time
from ray import serve

# The model registry: id -> checkpoint. The base model everyone shares, plus program specialists.
from aiddd_pipeline import OF3_P2_CKPT, OF3_FT_CKPT
CHECKPOINTS = {
    "base":    str(OF3_P2_CKPT),    # general OF3-p2 145k, the default for every program
    "ev71-v1": str(OF3_FT_CKPT),    # the EV-A71 program's specialist (notebook 02's fine-tune)
    "ev71-v2": str(OF3_FT_CKPT),    # a newer version of that specialist
}


@serve.deployment
class ModelPool:
    @serve.multiplexed(max_num_models_per_replica=2)   # keep 2 models hot per replica; evict the rest
    async def get_model(self, model_id: str):
        # Ray calls this on a cache miss, then caches the result. A real deployment loads weights
        # onto the GPU here; this walkthrough just records the load and returns the checkpoint.
        replica = serve.get_replica_context().replica_tag[:6]
        print(f"  [replica {replica}] loading {model_id}  <- {CHECKPOINTS[model_id].split('/')[-1]}")
        time.sleep(1)                                  # stand-in for reading weights onto the GPU
        return {"model_id": model_id, "checkpoint": CHECKPOINTS[model_id]}

    async def __call__(self, smiles: str) -> dict:
        model_id = serve.get_multiplexed_model_id()    # the id the caller asked for
        model = await self.get_model(model_id)         # cached per id on this replica
        replica = serve.get_replica_context().replica_tag[:6]
        return {"smiles": smiles, "served_by": model["model_id"], "replica": replica}

## 2. Run it and get a handle

`serve.run` starts the deployment in this Workspace and returns a `DeploymentHandle`. The same
`ModelPool.bind()` is what an Anyscale Service runs in production; here it runs locally so we can call it
from Python.

In [ ]:
handle = serve.run(ModelPool.bind(), name="multiplex-demo")
handle

## 3. Pick a model per request

`handle.options(multiplexed_model_id="base")` tells Ray which model this request wants. The router sends
it to a replica that already has that model loaded; if none does, a replica loads it (the `loading ...`
line). Sending the same id again is served from cache, with no load.

In [ ]:
for model_id in ["base", "ev71-v1", "base"]:
    out = await handle.options(multiplexed_model_id=model_id).remote("CSc1ncnc(...)")
    print(out)

`base` loads once, `ev71-v1` loads once, and the second `base` is served from cache (no `loading`
line). The multiplex id, not the SMILES, decides which model runs.

## 4. Eviction: `max_num_models_per_replica`

The pool keeps only 2 models per replica. Ask for a 3rd distinct id and Ray evicts the least-recently
used one. Request it again afterwards and it reloads, which the load log shows twice.

In [ ]:
for model_id in ["ev71-v2", "base"]:     # base was least-recently-used; v2 pushes it out
    out = await handle.options(multiplexed_model_id=model_id).remote("CSc1ncnc(...)")
    print(out)
# "base" prints a second "loading" line: it was evicted to make room for ev71-v2, then reloaded.

## 5. The HTTP equivalent

A non-Python client picks the model with the `serve_multiplexed_model_id` request header instead of
`handle.options(...)`:

```python
import requests
requests.post(SERVICE_URL + "/predict",
              json={"smiles": smiles},
              headers={"serve_multiplexed_model_id": "ev71-v2"})
```

Ray routes the request to a replica holding `ev71-v2`, exactly as the handle does.

## 6. The real thing

`serve_viewer_multi.py` is this exact pattern with OpenFold3:

- `OF3Pool.get_model` loads a real checkpoint onto half an L4 instead of a path string.
- A `ViewerMulti` deployment in front resolves a **program** to its current model id and calls the pool
  with `handle.options(multiplexed_model_id=...)`, then composes Boltz-2 affinity alongside the pose.
- Routing a program to a new version is a config change, so you **promote** a new specialist, **roll
  back**, or reach an old version **by id** for an audit, while the base model keeps serving everyone
  else. Notebook `03_serve.ipynb` walks that lifecycle.

Same API, same one GPU for the whole shop.

## 7. Stop the deployment

In [ ]:
serve.shutdown()